# 02 Cleaning

Clean both raw datasets independently and export processed files to `data/processed/`.

**Transformations**

*Close approaches*
- Parse `close_approach_date` to datetime
- Drop exact duplicates
- Trim whitespace on string columns
- Impute `velocity_infinity_km_s` and `absolute_magnitude` nulls with `velocity_km_s` / column median
- Sort chronologically

*Near-Earth Asteroids*
- Coerce `pdes` to string (mixed-type warning source)
- Parse `first_obs` / `last_obs` to datetime
- Drop rows with missing `moid_au` (critical hazard field)
- Normalize column names via `scripts.etl_pipeline.basic_clean`
- Keep sparse fields (`name`, `albedo`, `rot_per`) — they're measurement-limited, not errors


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import basic_clean

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## Dataset 1 — Close approaches

In [ ]:
close_raw = pd.read_csv(RAW_DIR / 'asteroid_close_approaches_2015_2035.csv')
print('Raw shape:', close_raw.shape)
close_raw.head(3)


In [ ]:
close = basic_clean(close_raw)

close['close_approach_date'] = pd.to_datetime(close['close_approach_date'], errors='coerce')

close['velocity_infinity_km_s'] = close['velocity_infinity_km_s'].fillna(close['velocity_km_s'])
close['absolute_magnitude'] = close['absolute_magnitude'].fillna(close['absolute_magnitude'].median())

close['approach_year'] = close['close_approach_date'].dt.year
close['approach_month'] = close['close_approach_date'].dt.month

close = close.sort_values('close_approach_date').reset_index(drop=True)

print('Cleaned shape:', close.shape)
print('Remaining nulls:')
print(close.isna().sum()[close.isna().sum() > 0])
close.head(3)


In [ ]:
CLOSE_OUT = PROCESSED_DIR / 'close_approaches_clean.csv'
close.to_csv(CLOSE_OUT, index=False)
print(f'Saved {len(close):,} rows -> {CLOSE_OUT}')


## Dataset 2 — Near-Earth Asteroids

In [ ]:
nea_raw = pd.read_csv(RAW_DIR / 'near_earth_asteroids_2025.csv', low_memory=False, dtype={'pdes': 'string'})
print('Raw shape:', nea_raw.shape)
nea_raw.head(3)


In [ ]:
nea = basic_clean(nea_raw)

nea['first_obs'] = pd.to_datetime(nea['first_obs'], errors='coerce')
nea['last_obs']  = pd.to_datetime(nea['last_obs'],  errors='coerce')

before = len(nea)
nea = nea.dropna(subset=['moid_au']).reset_index(drop=True)
print(f'Dropped {before - len(nea)} rows missing moid_au')

# Derived columns useful downstream
nea['is_named'] = nea['name'].notna()
nea['observation_span_years'] = (nea['last_obs'] - nea['first_obs']).dt.days / 365.25

print('Cleaned shape:', nea.shape)
nea.head(3)


In [ ]:
NEA_OUT = PROCESSED_DIR / 'near_earth_asteroids_clean.csv'
nea.to_csv(NEA_OUT, index=False)
print(f'Saved {len(nea):,} rows -> {NEA_OUT}')


## Sanity checks

In [ ]:
print('Close approaches — date range:', close['close_approach_date'].min(), 'to', close['close_approach_date'].max())
print('Close approaches — unique asteroids:', close['designation'].nunique())
print('NEA — PHA count:', int(nea['pha'].sum()), f"({nea['pha'].mean()*100:.2f}%)")
print('NEA — size categories:')
print(nea['size_category'].value_counts())
